# Análisis exploratorio de train

## Preparación

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from breast_cnn.configuration import load_phase_one_config
from breast_cnn.data.loaders import create_data_loaders
from breast_cnn.data.metadata import load_samples
from breast_cnn.data.reporting import run_train_eda

project_root = Path(os.environ.get("BREAST_CNN_ROOT", "..")).expanduser().resolve()
data_config, pipeline_config, eda_config = load_phase_one_config(project_root)
figure_directory = project_root / eda_config.figures_dir
eda_report = run_train_eda(project_root, data_config, eda_config)

## Balance de clases

In [ ]:
display(Image(filename=str(figure_directory / "class_balance_train_patient.png"), width=800))
display(Image(filename=str(figure_directory / "class_balance_train_slice.png"), width=800))

## Cortes por paciente

In [ ]:
display(Image(filename=str(figure_directory / "slices_per_patient_train_patient.png"), width=800))

## Cohortes

In [ ]:
display(Image(filename=str(figure_directory / "cohort_distribution_train_patient.png"), width=800))

## Valores ausentes

In [ ]:
display(Image(filename=str(figure_directory / "missing_values_train_patient.png"), width=800))
missing_by_cohort = pd.DataFrame(eda_report["tables"]["missing_by_cohort"]).set_index("data")
display(missing_by_cohort.loc[:, missing_by_cohort.sum().gt(0)])

## Variables clínicas y de adquisición

In [ ]:
numeric_summary = pd.DataFrame(eda_report["tables"]["numeric_summary"]).set_index("column")
display(numeric_summary.round(2))

## Subtipos por cohorte

In [ ]:
subtype_balance = pd.DataFrame(eda_report["tables"]["subtype_balance"])
display(subtype_balance.pivot(index=["data", "HR_HER2_STATUS"], columns="pCR", values="patients"))

## Distribución de folds

In [ ]:
fold_summary = pd.DataFrame(eda_report["folds"]).set_index("fold")
display(fold_summary)

## Fases DCE y realce

In [ ]:
for figure_path in eda_report["figures"]:
    if Path(figure_path).name.startswith("dce_phases_"):
        display(Image(filename=str(project_root / figure_path), width=1100))

## Lote de entrenamiento

In [ ]:
samples = load_samples(project_root, data_config.metadata_dir)
train_loader, validation_loader = create_data_loaders(
    samples,
    project_root,
    path_aliases=data_config.image_path_aliases,
    **pipeline_config.model_dump(exclude={"report_path"}),
)
train_batch = next(iter(train_loader))
print("Imágenes:", tuple(train_batch["image"].shape), train_batch["image"].dtype)
print("Rango:", float(train_batch["image"].min()), float(train_batch["image"].max()))
print("Etiquetas:", tuple(train_batch["label"].shape), train_batch["label"].dtype)